<a href="https://colab.research.google.com/github/Sugaminni/ML-Assignment-3/blob/main/ML_Assignment_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Part 1: Feedforward Neural Network

In [ ]:
# Imports For Dependencies
import numpy as np

# Imports data
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

Wine Dataset and Feedforward NN

In [ ]:
# Loads and split data
wine = load_wine()

# Retrieves data
X, y = wine.data, wine.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Standardizes features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Encodes labels
def one_hot(y, num_classes):
    out = np.zeros((len(y), num_classes))
    out[np.arange(len(y)), y] = 1
    return out

y_train_encoded = one_hot(y_train, 3)
y_test_encoded = one_hot(y_test, 3)

NN Setup and Training

In [ ]:
# Activation functions
def sigmoid(x):
    return 1 / (1 + np.exp(-x))
def sigmoid_derivative(x):
    return x * (1 - x)

# Network sizes (13 > 4 > 3 > 3)
np.random.seed(42)
W1 = np.random.randn(13, 4)
W2 = np.random.randn(4, 3)
W3 = np.random.randn(3, 3)

learning_rate = 0.01
epochs = 5000

# Trains with forward + backprop
for epoch in range(epochs):
    # Forward pass
    a1 = sigmoid(np.dot(X_train, W1))
    a2 = sigmoid(np.dot(a1, W2))
    output = sigmoid(np.dot(a2, W3))

    # Errors
    error = y_train_encoded - output

    # Backpropagation
    d_output = error * sigmoid_derivative(output)
    d_hidden2 = np.dot(d_output, W3.T) * sigmoid_derivative(a2)
    d_hidden1 = np.dot(d_hidden2, W2.T) * sigmoid_derivative(a1)

    # Updates weights
    W3 += learning_rate * np.dot(a2.T, d_output)
    W2 += learning_rate * np.dot(a1.T, d_hidden2)
    W1 += learning_rate * np.dot(X_train.T, d_hidden1)

Displays Test Accuracy

In [ ]:
# Forward pass on the test data
a1_test = sigmoid(np.dot(X_test, W1))
a2_test = sigmoid(np.dot(a1_test, W2))
output_test = sigmoid(np.dot(a2_test, W3))

# Shows Predicted classes and accuracy
y_pred = np.argmax(output_test, axis=1)
accuracy = np.mean(y_pred == y_test)

print("Wine Dataset Accuracy:", accuracy)

Part 2: CIFAR10 CNN with Keras

In [ ]:
# Imports For Dependencies
import numpy as np
from keras.datasets import cifar10
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from keras.utils import to_categorical
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

# Loads CIFAR10
(X, y) = cifar10.load_data()[0]
X_test_full, y_test_full = cifar10.load_data()[1]

Loads and Preprocesses Data

In [ ]:
# Combines train and test to create 70/30 split
X = np.concatenate((X, X_test_full), axis=0)
y = np.concatenate((y, y_test_full), axis=0)

# Normalizes images and splits 70/30
X = X.astype("float32") / 255.0
y = y.flatten()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Encodes labels
y_train_encoded = to_categorical(y_train, 10)
y_test_encoded = to_categorical(y_test, 10)

Builds, Trains, and Eval for CNN

In [ ]:
# Builds the CNN model
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(10, activation='softmax')
])

# Compiles and trains the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train_encoded, epochs=10, batch_size=64, verbose=1)

# Evaluates test set
loss, accuracy = model.evaluate(X_test, y_test_encoded, verbose=0)

# Predicts and computes the F1 score
y_pred = np.argmax(model.predict(X_test), axis=1)
f1 = f1_score(y_test, y_pred, average='weighted')

print("CIFAR10 Test Accuracy:", accuracy)
print("CIFAR10 F1 Score:", f1)